# Food Delivery BDA Project — Member 1: Preprocessing + HDFS

**Scope (per project brief):** own the pipeline from the raw Kaggle CSV → dataset inspection →
documented preprocessing decisions → cleaned dataset (`orders_clean.csv`) → HDFS storage → verification.

Dataset: [Food Delivery Order History Data](https://www.kaggle.com/datasets/sujalsuthar/food-delivery-order-history-data)
by Sujal Suthar — 21,321 records, 29 columns, six **imaginary** restaurants, Delhi NCR only.

> Run cells top to bottom. This notebook has been updated to run locally using the raw data file
> at `data/raw/order_history_kaggle_data.csv` and outputs to `data/processed/orders_clean.csv`.


## Part 0 — Setup Data Directory

The dataset is expected to be placed locally at `data/raw/order_history_kaggle_data.csv`.
If you haven't downloaded it yet, download it manually from Kaggle and place it in the folder `data/raw/`.


In [1]:
# Make sure the data folders exist
import os
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

raw_data_path = "data/raw/order_history_kaggle_data.csv"
if os.path.exists(raw_data_path):
    print(f"Raw dataset is correctly placed at {raw_data_path}")
else:
    print(f"WARNING: Raw dataset not found at {raw_data_path}. Please place the dataset there before running.")


Raw dataset is correctly placed at data/raw/order_history_kaggle_data.csv


In [2]:
# Verify raw file size and location
import os
raw_data_path = "data/raw/order_history_kaggle_data.csv"
if os.path.exists(raw_data_path):
    size_bytes = os.path.getsize(raw_data_path)
    print(f"File size: {size_bytes / (1024*1024):.2f} MB")
else:
    print("Raw file not found to inspect size.")


File size: 6.34 MB


## Part 1 — Dataset Inspection

Per the project brief: **inspect actual values before writing any cleaning logic.** Do not assume
nulls, timestamp format, or dtypes.


In [3]:
import pandas as pd
import numpy as np
import os

CSV_PATH = "data/raw/order_history_kaggle_data.csv"
if not os.path.exists(CSV_PATH):
    if os.path.exists("../data/raw/order_history_kaggle_data.csv"):
        CSV_PATH = "../data/raw/order_history_kaggle_data.csv"

print("Using CSV:", CSV_PATH)
df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape)
df.head()


Using CSV: data/raw/order_history_kaggle_data.csv
Shape: (21321, 29)


,Restaurant ID,Restaurant name,Subzone,City,Order ID,Order Placed At,Order Status,Delivery,Distance,Items in order,...,Rating,Review,Cancellation / Rejection reason,Restaurant compensation (Cancellation),Restaurant penalty (Rejection),KPT duration (minutes),Rider wait time (minutes),Order Ready Marked,Customer complaint tag,Customer ID
0,20320607,Swaad,Sector 4,Delhi NCR,6168884918,"11:38 PM, September 10 2024",Delivered,Zomato Delivery,3km,"1 x Grilled Chicken Jamaican Tender, 1 x Grill...",...,NaN,NaN,NaN,NaN,NaN,18.35,11.6,Correctly,NaN,5d6c2b96db963098bc69768bea504c8bf46106a8a5178e...
1,20320607,Swaad,Sector 4,Delhi NCR,6170707559,"11:34 PM, September 10 2024",Delivered,Zomato Delivery,2km,"1 x Peri Peri Fries, 1 x Fried Chicken Angara ...",...,NaN,NaN,NaN,NaN,NaN,16.95,3.6,Correctly,NaN,0781815deb4a10a574e9fee4fa0b86b074d4a0b36175d5...
2,20320607,Swaad,Sector 4,Delhi NCR,6169375019,"03:52 PM, September 10 2024",Delivered,Zomato Delivery,<1km,1 x Bone in Peri Peri Grilled Chicken,...,NaN,NaN,NaN,NaN,NaN,14.05,12.2,Correctly,NaN,f93362f5ce5382657482d164e368186bcec9c6225fd93d...
3,20320607,Swaad,Sector 4,Delhi NCR,6151677434,"03:45 PM, September 10 2024",Delivered,Zomato Delivery,2km,"1 x Fried Chicken Ghostbuster Tender, 1 x Anga...",...,4.0,NaN,NaN,NaN,NaN,19.00,3.3,Correctly,NaN,1ed226d1b8a5f7acee12fc1d6676558330a3b2b742af5d...
4,20320607,Swaad,Sector 4,Delhi NCR,6167540897,"03:04 PM, September 10 2024",Delivered,Zomato Delivery,2km,"1 x Peri Peri Krispers, 1 x Fried Chicken Anga...",...,NaN,NaN,NaN,NaN,NaN,15.97,1.0,Correctly,NaN,d21a2ac6ea06b31cc3288ab20c4ef2f292066c096f2c5f...


In [4]:
# Schema check — confirm all 29 expected columns exist, no accidental duplicates
expected_cols = [
    "Restaurant ID", "Restaurant name", "Subzone", "City", "Order ID", "Order Placed At",
    "Order Status", "Delivery", "Distance", "Items in order", "Instructions",
    "Discount construct", "Bill subtotal", "Packaging charges",
    "Restaurant discount (Promo)", "Restaurant discount (Flat offs, Freebies & others)",
    "Gold discount", "Brand pack discount", "Total", "Rating", "Review",
    "Cancellation / Rejection reason", "Restaurant compensation (Cancellation)",
    "Restaurant penalty (Rejection)", "KPT duration (minutes)", "Rider wait time (minutes)",
    "Order Ready Marked", "Customer complaint tag", "Customer ID"
]

print("Number of columns in file:", len(df.columns))
print("Actual columns:\n", list(df.columns))
print()
print("Duplicate column names:", df.columns[df.columns.duplicated()].tolist())

missing_expected = [c for c in expected_cols if c not in df.columns]
extra_actual = [c for c in df.columns if c not in expected_cols]
print("Expected columns NOT found in file:", missing_expected)
print("Columns in file NOT in expected list:", extra_actual)
print("(If either list is non-empty, update `expected_cols` to match the real header before proceeding —")
print(" this dataset's exact column names can vary slightly from the case-study description.)")


Number of columns in file: 29
Actual columns:
 ['Restaurant ID', 'Restaurant name', 'Subzone', 'City', 'Order ID', 'Order Placed At', 'Order Status', 'Delivery', 'Distance', 'Items in order', 'Instructions', 'Discount construct', 'Bill subtotal', 'Packaging charges', 'Restaurant discount (Promo)', 'Restaurant discount (Flat offs, Freebies & others)', 'Gold discount', 'Brand pack discount', 'Total', 'Rating', 'Review', 'Cancellation / Rejection reason', 'Restaurant compensation (Cancellation)', 'Restaurant penalty (Rejection)', 'KPT duration (minutes)', 'Rider wait time (minutes)', 'Order Ready Marked', 'Customer complaint tag', 'Customer ID']

Duplicate column names: []
Expected columns NOT found in file: []
Columns in file NOT in expected list: []
(If either list is non-empty, update `expected_cols` to match the real header before proceeding —
 this dataset's exact column names can vary slightly from the case-study description.)


In [5]:
# Data types as pandas inferred them
df.dtypes


Restaurant ID                                           int64
Restaurant name                                        object
Subzone                                                object
City                                                   object
Order ID                                                int64
Order Placed At                                        object
Order Status                                           object
Delivery                                               object
Distance                                               object
Items in order                                         object
Instructions                                           object
Discount construct                                     object
Bill subtotal                                         float64
Packaging charges                                     float64
Restaurant discount (Promo)                           float64
Restaurant discount (Flat offs, Freebies & others)    float64
Gold dis

In [6]:
# Missing value report for every column — counts and percentages
missing_report = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_pct": (df.isnull().sum() / len(df) * 100).round(2),
    "dtype": df.dtypes.astype(str)
}).sort_values("missing_count", ascending=False)
missing_report


,missing_count,missing_pct,dtype
Restaurant penalty (Rejection),21318,99.99,float64
Restaurant compensation (Cancellation),21188,99.38,float64
Cancellation / Rejection reason,21135,99.13,object
Review,21025,98.61,object
Customer complaint tag,20852,97.80,object
Instructions,20601,96.62,object
Rating,18830,88.32,float64
Discount construct,5498,25.79,object
KPT duration (minutes),295,1.38,float64
Rider wait time (minutes),168,0.79,float64


**Decide what missing means per field before filling anything** (brief explicitly says: do not
auto-fill with 0). Typical interpretation for this dataset — verify against the actual missing-value
report above:

| Field | Likely meaning of missing |
|---|---|
| Rating / Review | Customer didn't rate/review — not applicable, not zero |
| Cancellation / Rejection reason | Order wasn't cancelled/rejected — not applicable |
| Restaurant compensation / penalty | No cancellation/rejection event occurred — not applicable |
| Customer complaint tag | No complaint filed — not applicable |
| KPT duration / Rider wait time | Possibly not tracked for cancelled/rejected orders — check correlation with Order Status |
| Order Ready Marked | Possibly only recorded for delivered orders — check correlation with Order Status |

Confirm these by cross-tabulating against `Order Status` in the next cell rather than assuming.


In [7]:
# Verify the "not applicable" hypotheses above by cross-tabulating against Order Status
if "Order Status" in df.columns:
    for col in ["Rating", "Cancellation / Rejection reason", "KPT duration (minutes)",
                "Rider wait time (minutes)", "Order Ready Marked", "Customer complaint tag"]:
        if col in df.columns:
            print(f"--- {col}: missing rate by Order Status ---")
            print(df.groupby("Order Status")[col].apply(lambda s: s.isnull().mean().round(3)))
            print()


--- Rating: missing rate by Order Status ---
Order Status
Delivered           0.882
Picked up           1.000
Rejected            1.000
Return cancelled    1.000
Returned            1.000
Timed out           1.000
Name: Rating, dtype: float64

--- Cancellation / Rejection reason: missing rate by Order Status ---
Order Status
Delivered           1.0
Picked up           1.0
Rejected            0.0
Return cancelled    0.0
Returned            0.0
Timed out           1.0
Name: Cancellation / Rejection reason, dtype: float64

--- KPT duration (minutes): missing rate by Order Status ---
Order Status
Delivered           0.009
Picked up           0.000
Rejected            0.614
Return cancelled    0.000
Returned            0.000
Timed out           1.000
Name: KPT duration (minutes), dtype: float64

--- Rider wait time (minutes): missing rate by Order Status ---
Order Status
Delivered           0.002
Picked up           0.000
Rejected            0.747
Return cancelled    0.000
Returned         

In [8]:
# Numeric field sanity checks — non-numeric strings, negatives, impossible ratings, outliers
numeric_candidates = [
    "Bill subtotal", "Packaging charges", "Restaurant discount (Promo)",
    "Restaurant discount (Flat offs, Freebies & others)", "Gold discount",
    "Brand pack discount", "Total", "Rating", "Restaurant compensation (Cancellation)",
    "Restaurant penalty (Rejection)", "KPT duration (minutes)", "Rider wait time (minutes)"
]
numeric_candidates = [c for c in numeric_candidates if c in df.columns]

for col in numeric_candidates:
    coerced = pd.to_numeric(df[col], errors="coerce")
    non_numeric_count = coerced.isnull().sum() - df[col].isnull().sum()  # became NaN but wasn't originally
    print(f"{col}: non-numeric entries = {non_numeric_count}, "
          f"min = {coerced.min()}, max = {coerced.max()}, negatives = {(coerced < 0).sum()}")

print()
if "Rating" in df.columns:
    r = pd.to_numeric(df["Rating"], errors="coerce")
    print("Rating out-of-range (not 1-5):", ((r < 1) | (r > 5)).sum())


Bill subtotal: non-numeric entries = 0, min = 50.0, max = 16080.0, negatives = 0
Packaging charges: non-numeric entries = 0, min = 0.0, max = 603.0, negatives = 0
Restaurant discount (Promo): non-numeric entries = 0, min = 0.0, max = 4020.0, negatives = 0
Restaurant discount (Flat offs, Freebies & others): non-numeric entries = 0, min = 0.0, max = 7787.0, negatives = 0
Gold discount: non-numeric entries = 0, min = 0.0, max = 280.1, negatives = 0
Brand pack discount: non-numeric entries = 0, min = 0.0, max = 554.8, negatives = 0
Total: non-numeric entries = 0, min = 52.5, max = 12663.0, negatives = 0
Rating: non-numeric entries = 0, min = 1.0, max = 5.0, negatives = 0
Restaurant compensation (Cancellation): non-numeric entries = 0, min = 83.58, max = 3236.98, negatives = 0
Restaurant penalty (Rejection): non-numeric entries = 0, min = 0.0, max = 0.0, negatives = 0
KPT duration (minutes): non-numeric entries = 0, min = 0.0, max = 90.87, negatives = 0
Rider wait time (minutes): non-numeri

In [9]:
# Timestamp inspection — DO NOT assume the format
if "Order Placed At" in df.columns:
    print("Sample raw values:")
    print(df["Order Placed At"].dropna().head(10).tolist())
    print()
    parsed = pd.to_datetime(df["Order Placed At"], errors="coerce")
    print("Unparseable timestamps:", parsed.isnull().sum() - df["Order Placed At"].isnull().sum())
    print("Date range:", parsed.min(), "to", parsed.max())


Sample raw values:
['11:38 PM, September 10 2024', '11:34 PM, September 10 2024', '03:52 PM, September 10 2024', '03:45 PM, September 10 2024', '03:04 PM, September 10 2024', '12:28 PM, September 10 2024', '12:03 AM, September 10 2024', '10:54 PM, September 09 2024', '10:51 PM, September 09 2024', '03:22 PM, September 09 2024']



C:\Users\adarsh p\AppData\Local\Temp\ipykernel_17648\1500348731.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df["Order Placed At"], errors="coerce")


Unparseable timestamps: 0
Date range: 2024-09-01 00:13:00 to 2025-01-31 23:59:00


In [10]:
# Categorical field unique-value inspection — don't normalise until you've seen the real values
categorical_candidates = ["Restaurant name", "City", "Subzone", "Order Status", "Delivery",
                           "Distance", "Order Ready Marked", "Customer complaint tag",
                           "Cancellation / Rejection reason"]
categorical_candidates = [c for c in categorical_candidates if c in df.columns]

for col in categorical_candidates:
    vals = df[col].value_counts(dropna=False)
    print(f"--- {col} ({df[col].nunique(dropna=True)} unique, non-null) ---")
    print(vals.head(15))
    print()


--- Restaurant name (6 unique, non-null) ---
Restaurant name
Aura Pizzas             14548
Swaad                    6332
Dilli Burger Adda         227
Tandoori Junction         154
The Chicken Junction       32
Masala Junction            28
Name: count, dtype: int64

--- City (1 unique, non-null) ---
City
Delhi NCR    21321
Name: count, dtype: int64

--- Subzone (8 unique, non-null) ---
Subzone
Greater Kailash 2 (GK2)    7380
Sector 4                   6530
DLF Phase 1                3686
Sector 135                 2442
Vasant Kunj                 920
Shahdara                    360
Chittaranjan Park             2
Sikandarpur                   1
Name: count, dtype: int64

--- Order Status (6 unique, non-null) ---
Order Status
Delivered           21131
Rejected              158
Returned               25
Return cancelled        3
Picked up               3
Timed out               1
Name: count, dtype: int64

--- Delivery (1 unique, non-null) ---
Delivery
Zomato Delivery    21321
Name: cou

In [11]:
# Text fields that may contain commas/quotes — flag rows where that's true, since it affects CSV export
text_fields = ["Items in order", "Instructions", "Discount construct", "Review",
               "Cancellation / Rejection reason", "Customer complaint tag"]
text_fields = [c for c in text_fields if c in df.columns]

for col in text_fields:
    has_comma = df[col].astype(str).str.contains(",").sum()
    has_quote = df[col].astype(str).str.contains('"').sum()
    print(f"{col}: rows containing a comma = {has_comma}, rows containing a quote char = {has_quote}")


Items in order: rows containing a comma = 11694, rows containing a quote char = 0
Instructions: rows containing a comma = 87, rows containing a quote char = 1
Discount construct: rows containing a comma = 0, rows containing a quote char = 0
Review: rows containing a comma = 63, rows containing a quote char = 0
Cancellation / Rejection reason: rows containing a comma = 0, rows containing a quote char = 0
Customer complaint tag: rows containing a comma = 0, rows containing a quote char = 0


## Part 2 — Preprocessing

Based on the inspection above, this section documents and applies the cleaning steps. **Re-check the
printed output from Part 1 on your actual download and adjust the logic below if reality differs** —
this is written to be robust to it, but the brief is explicit that assumptions aren't allowed.

Cleaning operations applied:
1. Strip whitespace from all string/object columns.
2. Coerce numeric columns to actual numeric dtype (invalid strings → NaN, not dropped silently — logged).
3. Parse `Order Placed At` into a proper timestamp column; keep the original string as a backup column.
4. Leave "not applicable" missing values as NaN (do NOT convert to 0) — Hive can read these as NULL.
5. Flag (not delete) rows with negative values in financial fields, for manual review.
6. Keep all 29 columns — no analytical columns are dropped without justification.
7. Save to a **separate** file `orders_clean.csv`, leaving the raw file untouched.


In [12]:
df_clean = df.copy()

# 1. Strip whitespace from string columns
obj_cols = df_clean.select_dtypes(include="object").columns
for col in obj_cols:
    df_clean[col] = df_clean[col].astype(str).where(df_clean[col].notnull(), df_clean[col])
    df_clean[col] = df_clean[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

# 2. Coerce numeric columns properly, logging how many values failed to parse
numeric_fail_log = {}
for col in numeric_candidates:
    original_non_null = df_clean[col].notnull().sum()
    coerced = pd.to_numeric(df_clean[col], errors="coerce")
    newly_null = coerced.isnull().sum() - df_clean[col].isnull().sum()
    numeric_fail_log[col] = int(newly_null)
    df_clean[col] = coerced

print("Values that failed numeric coercion per column (investigate before trusting them as NaN):")
print(numeric_fail_log)


Values that failed numeric coercion per column (investigate before trusting them as NaN):
{'Bill subtotal': 0, 'Packaging charges': 0, 'Restaurant discount (Promo)': 0, 'Restaurant discount (Flat offs, Freebies & others)': 0, 'Gold discount': 0, 'Brand pack discount': 0, 'Total': 0, 'Rating': 0, 'Restaurant compensation (Cancellation)': 0, 'Restaurant penalty (Rejection)': 0, 'KPT duration (minutes)': 0, 'Rider wait time (minutes)': 0}


In [13]:
# 3. Parse the timestamp, keep the original as backup
if "Order Placed At" in df_clean.columns:
    df_clean["Order Placed At (raw)"] = df_clean["Order Placed At"]
    df_clean["Order Placed At"] = pd.to_datetime(df_clean["Order Placed At"], errors="coerce")
    print("Timestamps that failed to parse:",
          df_clean["Order Placed At"].isnull().sum() - df["Order Placed At"].isnull().sum())


C:\Users\adarsh p\AppData\Local\Temp\ipykernel_17648\3209489144.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean["Order Placed At"] = pd.to_datetime(df_clean["Order Placed At"], errors="coerce")


Timestamps that failed to parse: 0


In [14]:
# 4. Leave 'not applicable' fields as NaN — nothing to do here, just confirming we haven't filled anything
print("Missing counts preserved (should match Part 1 report, modulo any newly-coerced numeric NaNs):")
print(df_clean.isnull().sum().sort_values(ascending=False).head(15))


Missing counts preserved (should match Part 1 report, modulo any newly-coerced numeric NaNs):
Restaurant penalty (Rejection)            21318
Restaurant compensation (Cancellation)    21188
Cancellation / Rejection reason           21135
Review                                    21025
Customer complaint tag                    20852
Instructions                              20601
Rating                                    18830
Discount construct                         5498
KPT duration (minutes)                      295
Rider wait time (minutes)                   168
Subzone                                       0
City                                          0
Restaurant ID                                 0
Restaurant name                               0
Packaging charges                             0
dtype: int64


In [15]:
# 5. Flag negative values in financial fields for manual review (kept, not deleted)
financial_cols = ["Bill subtotal", "Packaging charges", "Total",
                   "Restaurant discount (Promo)", "Restaurant discount (Flat offs, Freebies & others)",
                   "Gold discount", "Brand pack discount"]
financial_cols = [c for c in financial_cols if c in df_clean.columns]

neg_mask = pd.Series(False, index=df_clean.index)
for col in financial_cols:
    neg_mask |= (df_clean[col] < 0)

df_clean["flag_negative_financial"] = neg_mask
print("Rows flagged with a negative financial value:", neg_mask.sum())


Rows flagged with a negative financial value: 0


In [16]:
# 6/7. Save the cleaned dataset separately — raw file is untouched
OUTPUT_PATH = "data/processed/orders_clean.csv"
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df_clean.to_csv(OUTPUT_PATH, index=False)
print(f"Saved cleaned dataset to {OUTPUT_PATH}")
print("Shape:", df_clean.shape)


Saved cleaned dataset to data/processed/orders_clean.csv
Shape: (21321, 31)


### Before / after example (for the review demo)

Shows one raw record next to its cleaned counterpart — useful evidence for Member 1's demonstration.


In [17]:
idx = df.index[0]
print("RAW record:")
print(df.loc[idx])
print()
print("CLEANED record:")
print(df_clean.loc[idx])


RAW record:
Restaurant ID                                                                                  20320607
Restaurant name                                                                                   Swaad
Subzone                                                                                        Sector 4
City                                                                                          Delhi NCR
Order ID                                                                                     6168884918
Order Placed At                                                             11:38 PM, September 10 2024
Order Status                                                                                  Delivered
Delivery                                                                                Zomato Delivery
Distance                                                                                            3km
Items in order                                      

## Part 3 — HDFS (Docker-based Hadoop Stack)

Since the team uses a shared Docker Compose stack for Hadoop + Hive, HDFS commands are executed
against the `namenode` container. Below are the steps and CLI commands to format the namenode,
verify the status of the containers, and upload/manage the cleaned CSV in HDFS.


In [18]:
# Check if Docker compose containers are running
import subprocess
try:
    print("Checking running containers...")
    res = subprocess.run(["docker", "compose", "ps"], capture_output=True, text=True, shell=True)
    print(res.stdout)
    if "namenode" not in res.stdout:
        print("WARNING: 'namenode' service not found. Make sure you ran 'docker compose up -d' first.")
except Exception as e:
    print("Could not run docker command. Make sure Docker is installed and running:", e)


Checking running containers...
NAME      IMAGE     COMMAND   SERVICE   CREATED   STATUS    PORTS



In [19]:
# Copy the cleaned file to the NameNode container and load it into HDFS
import subprocess
import os

clean_csv_path = "data/processed/orders_clean.csv"
if not os.path.exists(clean_csv_path):
    if os.path.exists("../data/processed/orders_clean.csv"):
        clean_csv_path = "../data/processed/orders_clean.csv"

try:
    # 1. Ensure target HDFS directory exists
    print("Creating HDFS directory /food_delivery/input ...")
    subprocess.run("docker compose exec -T namenode hdfs dfs -mkdir -p /food_delivery/input", shell=True, check=True)
    
    # 2. Copy the local clean file into the namenode container temp space
    print("Copying local orders_clean.csv to NameNode container...")
    subprocess.run(f"docker compose cp {clean_csv_path} namenode:/tmp/orders_clean.csv", shell=True, check=True)
    
    # 3. Import from container temp space into HDFS
    print("Loading file from container temp space into HDFS...")
    subprocess.run("docker compose exec -T namenode hdfs dfs -put -f /tmp/orders_clean.csv /food_delivery/input/orders_clean.csv", shell=True, check=True)
    
    # 4. Clean up temp container space
    subprocess.run("docker compose exec -T namenode rm /tmp/orders_clean.csv", shell=True, check=True)
    
    print("Successfully uploaded orders_clean.csv to HDFS!")
except subprocess.CalledProcessError as err:
    print(f"Error executing docker commands: {err}")


Creating HDFS directory /food_delivery/input ...
Error executing docker commands: Command 'docker compose exec -T namenode hdfs dfs -mkdir -p /food_delivery/input' returned non-zero exit status 1.


In [20]:
# Verify HDFS files listing and disk usage
import subprocess
try:
    print("Listing /food_delivery/input content:")
    subprocess.run("docker compose exec -T namenode hdfs dfs -ls -h /food_delivery/input", shell=True)
    print("\nDisk usage in HDFS:")
    subprocess.run("docker compose exec -T namenode hdfs dfs -du -h /food_delivery/input", shell=True)
except Exception as e:
    print("Could not execute verification command on docker namenode:", e)


Listing /food_delivery/input content:

Disk usage in HDFS:


In [21]:
# Optional sanity check: read the first few lines straight back out of HDFS
import subprocess
try:
    print("First 5 lines of orders_clean.csv in HDFS:")
    res = subprocess.run("docker compose exec -T namenode hdfs dfs -cat /food_delivery/input/orders_clean.csv", shell=True, capture_output=True, text=True)
    lines = res.stdout.splitlines()
    for line in lines[:5]:
        print(line)
except Exception as e:
    print("Could not read from HDFS:", e)


First 5 lines of orders_clean.csv in HDFS:


### Handoff to Members 2 & 3

- **HDFS input path:** `/food_delivery/input/orders_clean.csv`
- **Cleaned schema:** same 29 original columns + `Order Placed At (raw)` (backup string) +
  `flag_negative_financial` (boolean flag, not a deletion).
- **Missing-value convention:** NaN in fields like Rating, Review, Cancellation reason, compensation,
  penalty, complaint tag means "not applicable" (no such event occurred) — not zero. Downstream Hive
  `NULL` handling should follow this, and MapReduce jobs should skip/ignore rather than treat NaN as 0
  when averaging.
- **Timestamp:** `Order Placed At` is parsed to a real datetime; confirm the exact format printed in
  Part 1 before Member 3 defines the Hive column type (`TIMESTAMP` vs `STRING`).

### For your review demo
Screenshot the outputs of: schema check, missing-value report, before/after record comparison, the
`hdfs dfs -ls` and `-du` cells above — those directly cover the "Execution screenshots" and
"HDFS storage demonstration" requirements from the brief.
